In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Import Libraries

In [ ]:
# CELL 1: Import Libraries
import pandas as pd
import numpy as np
import ast
import re
import requests
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from io import BytesIO
from difflib import get_close_matches
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download('punkt')
from nltk.stem.porter import PorterStemmer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


#Load Dataset

In [ ]:
# CELL 2: Load Dataset
dataset_path = "/content/drive/MyDrive/Machine Learning Project/Projects/Movie Recomendetion/extracted_dataset/"

movies = pd.read_csv(dataset_path + "tmdb_5000_movies.csv")
credits = pd.read_csv(dataset_path + "tmdb_5000_credits.csv")

print("Movies shape:", movies.shape)
print("Credits shape:", credits.shape)

Movies shape: (4803, 20)
Credits shape: (4803, 4)


#Merge Movies and Credits

In [ ]:
# CELL 3: Merge Movies and Credits
movies = movies.merge(credits, left_on='id', right_on='movie_id')
movies = movies[['movie_id', 'title_x', 'overview', 'genres', 'keywords', 'cast', 'crew', 'vote_average', 'vote_count', 'release_date']]
movies.rename(columns={'title_x': 'title'}, inplace=True)

print("Merged shape:", movies.shape)
movies.head(2)

Merged shape: (4803, 10)


,movie_id,title,overview,genres,keywords,cast,crew,vote_average,vote_count,release_date
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,11800,2009-12-10
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,4500,2007-05-19


#Drop Missing Values

In [ ]:
# CELL 4: Drop Missing Values
print("Missing values before:\n", movies.isnull().sum())

movies.dropna(subset=['overview'], inplace=True)

print("\nShape after dropping missing overview:", movies.shape)

Missing values before:
 movie_id        0
title           0
overview        3
genres          0
keywords        0
cast            0
crew            0
vote_average    0
vote_count      0
release_date    1
dtype: int64

Shape after dropping missing overview: (4800, 10)


#Parse JSON Columns (genres, keywords, cast, crew)

In [ ]:
# CELL 5: Parse JSON Columns
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

def convert_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
        counter += 1
    return L

def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert_cast)
movies['crew'] = movies['crew'].apply(fetch_director)

movies.head(2)

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,vote_count,release_date
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],7.2,11800,2009-12-10
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],6.9,4500,2007-05-19


#Build Tags Column

In [ ]:
# CELL 6: Build Tags Column
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

movies['overview_list'] = movies['overview'].apply(lambda x: x.split())

movies['tags'] = movies['overview_list'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

movies[['title', 'tags']].head(2)

,title,tags
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."


#Clean, Lowercase and Stem Tags

In [ ]:
# CELL 7: Clean, Lowercase and Stem Tags
ps = PorterStemmer()

def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

# create final working dataframe as independent copy
new_df = movies[['movie_id', 'title', 'tags', 'vote_average', 'vote_count', 'release_date']].copy()

new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

# remove punctuation, keep only letters, numbers and spaces
new_df['tags'] = new_df['tags'].apply(lambda x: re.sub(r'[^a-z0-9\s]', ' ', x))
new_df['tags'] = new_df['tags'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

new_df['tags'] = new_df['tags'].apply(stem)

# add display-friendly info columns
new_df['genres'] = movies['genres'].apply(lambda x: ', '.join(x))
new_df['cast'] = movies['cast'].apply(lambda x: ', '.join(x))
new_df['crew'] = movies['crew'].apply(lambda x: ', '.join(x))

new_df.head(2)

,movie_id,title,tags,vote_average,vote_count,release_date,genres,cast,crew
0,19995,Avatar,in the 22nd centuri a parapleg marin is dispat...,7.2,11800,2009-12-10,"Action, Adventure, Fantasy, ScienceFiction","SamWorthington, ZoeSaldana, SigourneyWeaver",JamesCameron
1,285,Pirates of the Caribbean: At World's End,captain barbossa long believ to be dead ha com...,6.9,4500,2007-05-19,"Adventure, Fantasy, Action","JohnnyDepp, OrlandoBloom, KeiraKnightley",GoreVerbinski


#TF-IDF Vectorization and Cosine Similarity

In [ ]:
# CELL 8: TF-IDF Vectorization and Cosine Similarity
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_vectors = tfidf.fit_transform(new_df['tags']).toarray()

tfidf_similarity = cosine_similarity(tfidf_vectors)

print("Similarity matrix shape:", tfidf_similarity.shape)

Similarity matrix shape: (4800, 4800)


#Weighted Rating (IMDB Formula)

In [ ]:
# CELL 9: Weighted Rating (IMDB Formula)
C = new_df['vote_average'].mean()
m = new_df['vote_count'].quantile(0.75)

def weighted_rating(row, m=m, C=C):
    v = row['vote_count']
    R = row['vote_average']
    return (v / (v + m) * R) + (m / (v + m) * C)

new_df['weighted_rating'] = new_df.apply(weighted_rating, axis=1)

print("C:", C, "| m:", m)
new_df[['title', 'vote_average', 'weighted_rating']].head(2)

C: 6.0929166666666665 | m: 737.25


,title,vote_average,weighted_rating
0,Avatar,7.2,7.134898
1,Pirates of the Caribbean: At World's End,6.9,6.786387


#TMDB Poster Fetch Function


In [ ]:
# CELL 10: TMDB Poster Fetch Function
TMDB_API_KEY = "Enter Your TMDB API Key"

def fetch_poster(movie_id):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={TMDB_API_KEY}&language=en-US"
    response = requests.get(url)
    data = response.json()
    poster_path = data.get('poster_path')
    if poster_path:
        return "https://image.tmdb.org/t/p/w500" + poster_path
    return None

# quick test
print(fetch_poster(19995))

https://image.tmdb.org/t/p/w500/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg


#Flexible Movie Search Function

In [ ]:
# CELL 11: Flexible Movie Search Function
movie_aliases = {
    "lotr": "The Lord of the Rings",
    "got": "Game of Thrones",
    "poc": "Pirates of the Caribbean",
    "hp": "Harry Potter",
    "tdkr": "The Dark Knight Rises",
}

def find_movie_title(user_input):
    titles = new_df['title'].tolist()
    titles_lower = [t.lower() for t in titles]
    user_input_lower = user_input.lower().strip()

    if user_input_lower in movie_aliases:
        user_input_lower = movie_aliases[user_input_lower].lower()

    if user_input_lower in titles_lower:
        return titles[titles_lower.index(user_input_lower)]

    substring_matches = [t for t in titles if user_input_lower in t.lower()]
    if substring_matches:
        return substring_matches[0]

    matches = get_close_matches(user_input_lower, titles_lower, n=1, cutoff=0.6)
    if matches:
        return titles[titles_lower.index(matches[0])]

    return None

#Final Recommend Function (Poster + Full Info Card)

In [ ]:
# CELL 12: Final Recommend Function (with Similarity Score and Genre Filter)
def show_recommendations(user_input, top_n=5, candidate_pool=30, genre_filter=None):
    movie = find_movie_title(user_input)

    if movie is None:
        print(f"'{user_input}' not found in dataset. Try another spelling.")
        return

    print(f"Recommendations based on: {movie}")
    if genre_filter:
        print(f"Genre filter applied: {genre_filter}")
    print()

    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = tfidf_similarity[movie_index]

    similar_indices = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:candidate_pool + 1]

    candidate_rows = []
    for i, score in similar_indices:
        row = new_df.iloc[i].copy()
        row['similarity_score'] = score
        candidate_rows.append(row)

    candidates = pd.DataFrame(candidate_rows)

    # apply genre filter if specified
    if genre_filter:
        candidates = candidates[candidates['genres'].str.contains(genre_filter, case=False, na=False)]

    if candidates.empty:
        print(f"No movies found with genre '{genre_filter}' in top candidates. Try a different genre or remove filter.")
        return

    candidates = candidates.sort_values('weighted_rating', ascending=False).head(top_n)

    fig, axes = plt.subplots(1, len(candidates), figsize=(24, 8))
    if len(candidates) == 1:
        axes = [axes]

    for ax, (idx, row) in zip(axes, candidates.iterrows()):
        poster_url = fetch_poster(row['movie_id'])
        if poster_url:
            img_data = requests.get(poster_url).content
            img = mpimg.imread(BytesIO(img_data), format='jpg')
            ax.imshow(img)

        info_text = (
            f"{row['title']}\n"
            f"Similarity: {row['similarity_score'] * 100:.1f}%\n"
            f"Rating: {row['vote_average']}/10\n"
            f"Release: {row['release_date']}\n"
            f"Genre: {row['genres']}\n"
            f"Director: {row['crew']}"
        )
        ax.set_title(info_text, fontsize=8, loc='left')
        ax.axis('off')

    plt.tight_layout()
    plt.show()

    return candidates

#Testing


In [ ]:
# CELL 13: Run Interactive Test
movie_input = input("Enter movie name: ")
genre_input = input("Enter genre filter (press Enter to skip): ").strip()

if genre_input:
    recs = show_recommendations(movie_input, genre_filter=genre_input)
else:
    recs = show_recommendations(movie_input)

In [ ]:
# CELL 14: Genre Overlap Evaluation Metric
def genre_overlap_score(source_title, recommended_df):
    source_genres = set(new_df[new_df['title'] == source_title]['genres'].values[0].split(', '))

    scores = []
    for _, row in recommended_df.iterrows():
        rec_genres = set(row['genres'].split(', '))
        overlap = len(source_genres & rec_genres)
        union = len(source_genres | rec_genres)
        jaccard = overlap / union if union > 0 else 0
        scores.append(jaccard)

    avg_score = sum(scores) / len(scores) if scores else 0
    return avg_score, scores

# find the actual matched title again for evaluation
matched_title = find_movie_title(movie_input)
avg_score, individual_scores = genre_overlap_score(matched_title, recs)

print(f"\nGenre Overlap Score for '{matched_title}': {avg_score:.3f} ({avg_score*100:.1f}%)")
print("Individual scores:", [f"{s:.2f}" for s in individual_scores])


Genre Overlap Score for 'The Lord of the Rings: The Fellowship of the Ring': 0.123 (12.3%)
Individual scores: ['0.20', '0.00', '0.17', '0.00', '0.25']


In [ ]:
# CELL 15: Genre Overlap Score Comparison across Multiple Movies
test_movies_list = [
    'Avatar',
    'The Dark Knight Rises',
    'Toy Story',
    'The Notebook',
    'The Lord of the Rings: The Fellowship of the Ring'
]

movie_names = []
overlap_scores = []

for m in test_movies_list:
    matched = find_movie_title(m)
    if matched is None:
        continue
    recs = show_recommendations(matched)
    avg_score, _ = genre_overlap_score(matched, recs)
    movie_names.append(matched)
    overlap_scores.append(avg_score)

# plot comparison bar chart
plt.figure(figsize=(10, 6))
bars = plt.bar(movie_names, overlap_scores, color='skyblue', edgecolor='black')
plt.ylabel('Genre Overlap Score (Jaccard)')
plt.title('Genre Overlap Score across Different Movies')
plt.xticks(rotation=30, ha='right')
plt.ylim(0, 1)

for bar, score in zip(bars, overlap_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{score:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()